## Stage 2.1 — Candidate Skill Retrieval (preferredLabel index)

**Role:** First notebook of Stage 2. Reads cleaned job posting pickles from Stage 1, encodes each description with a multilingual sentence-transformer model, and retrieves the top-K most similar ESCO skills using cosine similarity against a pre-encoded skill index.

**Position:** Run after `stage_1_read_initial_data_fast.ipynb` (Stage 1) and before `stage_2_2_add_romote_jobs.ipynb` (Stage 2.2).

**Output:** Enriched pickle files with `skill_ids` and `skill_labels` columns added, saved to `data/data-pipeline/stage_02/output/`.

# 2. Stage 2. Candidate Skill Retrieval

It reads **`stage1\output\***.pkl`** (output of the previous step), embeds each posting’s description with the appropriate *sentence‑transformer* model, retrieves the top‑K most similar ESCO **skill IDs**, and writes to **`stage2\output\****.pkl`**

## 2.1. Load packages and configuration

Load autoreload so changes to shared modules are picked up automatically. Import `general` for configuration and path management, and `stage1` for shared utilities. Call `clean_memory()` to release leftover variables from previous runs.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
import os
g.clean_memory()
import stage1 as st1

Import numerical, data, and NLP libraries. `SentenceTransformer` provides the multilingual embedding model. `tqdm` renders a progress bar during the per-record skill retrieval loop.

In [ ]:
import os, pathlib, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os, pathlib, re
from typing import Dict, List, Tuple
from sentence_transformers import SentenceTransformer, util

# Це код для коригування унікальних ID, що записалися на етапі 1

Define the Stage 2 process tracker builder. Reads the Stage 1 tracker and registers every file with `clean_status == complete` that is not yet present in the Stage 2 tracker. Creates a fresh Stage 2 tracker if none exists. Saves the updated tracker to disk.

In [ ]:
def get_stage_process_df(stage1_path, stage2_path):

    is_updated: bool = False

    if not os.path.exists(stage2_path):
        columns = ["input_file", "clean_path", "extract_path", "extract_status"]
        stage2_process = pd.DataFrame(columns=columns).astype(str)
        is_updated = True
    else:
        stage2_process = pd.read_pickle(stage2_path)

    stage1_process = pd.read_pickle(stage1_path)

    for _, row in stage1_process.iterrows():
        if row["clean_status"] == "complete" and stage2_process.loc[stage2_process["input_file"] == row["input_file"]].empty:
            new_row = pd.DataFrame([{'input_file': row['input_file'], 'clean_path': row['clean_path']}])
            stage2_process = pd.concat([stage2_process, new_row], ignore_index=True)
            is_updated = True

    if is_updated:
        stage2_process = stage2_process.sort_values(by="input_file").reset_index(drop=True)
        stage2_process.to_pickle(stage2_path)

    return stage2_process

Compile the sentence-splitting regex used by `SkillRetriever.retrieve()`. Splits text on sentence-ending punctuation followed by whitespace, allowing each sentence to be encoded and matched independently.

In [ ]:
import re
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

Compile the sentence-splitting regex used by `SkillRetriever.retrieve()`. Splits text on sentence-ending punctuation followed by whitespace, allowing each sentence to be encoded and matched independently.

In [ ]:
class SkillRetriever:
    def __init__(self, skills_dir: str, top_k: int = 25):
        self.dir    = pathlib.Path(skills_dir)
        self.k      = top_k
        self.vecs   : Dict[str, np.ndarray] = {}
        self.ids    : Dict[str, List[str]]  = {}
        self.labels : Dict[str, List[str]]  = {}
        self.embed  : Dict[str, SentenceTransformer] = {}
         # AltLabels index
        self.alt_vecs   : Dict[str, np.ndarray] = {}
        self.alt_ids    : Dict[str, List[str]]  = {}
        self.alt_labels : Dict[str, List[str]]  = {}
        self.alt_texts  : Dict[str, List[str]]  = {}

    def _embedder(self, lang: str):
        mid = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
        if mid not in self.embed:
            print(f"🔹 loading model {mid} for {lang}")
            self.embed[mid] = SentenceTransformer(mid)
        return self.embed[mid]

    def _skill_index(self, lang: str):
        if lang in self.vecs:
            return self.vecs[lang], self.ids[lang], self.labels[lang]
        csv = self.dir / f"skills_{lang}.csv"
        if not csv.exists():
            csv = self.dir / "skills_en.csv"; lang = "en"
        df = pd.read_csv(csv, dtype=str)
        emb = self._embedder(lang)
        print(f"   ↳ encoding {len(df):,} skills for {lang}")
        V = emb.encode(df["preferredLabel"].tolist(), batch_size=128,
                       normalize_embeddings=True, convert_to_numpy=True,
                       show_progress_bar=True)
        self.vecs[lang], self.ids[lang], self.labels[lang] = (
            V, df["conceptUri"].tolist(), df["preferredLabel"].tolist()
        )
        return V, self.ids[lang], self.labels[lang]

    def retrieve(self, lang: str, text: str) -> pd.DataFrame:
        """Return a *pandas DataFrame* with columns `skill_id`, `skill_label`."""
        V, sids, slabels = self._skill_index(lang)
        emb = self._embedder(lang)
        sent_vec = emb.encode(
            _SENT_SPLIT.split(text.strip()),
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        sims = np.dot(sent_vec, V.T)
        best_idx = np.argpartition(-sims, self.k, axis=1)[:, : self.k]
        # Flatten & de‑duplicate preserving score order
        idx_score_pairs = []
        for row, idxs in zip(sims, best_idx):
            for j in idxs:
                idx_score_pairs.append((j, row[j]))
        # Sort globally by similarity and pick top‑K unique skill IDs
        idx_score_pairs = sorted(idx_score_pairs, key=lambda x: -x[1])
        seen, rows = set(), []
        THRESH = 0.50  # minimum cosine‑similarity
        for j, score in idx_score_pairs:
            if score < THRESH:
                continue  # skip weak matches
            sid = sids[j]
            if sid not in seen:
                rows.append({"skill_id": sid, "skill_label": slabels[j]})
                seen.add(sid)
            if len(rows) == self.k:
                break
        return pd.DataFrame(rows)

    def _skill_index_alt(self, lang: str):
        """
        Build and cache an index over altLabels. Each alt label points to the same conceptUri.
        Returns (V_alt, alt_ids, alt_pref_labels).
        """
        if lang in self.alt_vecs:
            return self.alt_vecs[lang], self.alt_ids[lang], self.alt_labels[lang]

        csv = self.dir / f"skills_{lang}.csv"
        if not csv.exists():
            csv = self.dir / "skills_en.csv"; lang = "en"
        df = pd.read_csv(csv, dtype=str)

        # Prepare expanded rows for alt labels
        df["altLabels"] = df["altLabels"].fillna("")
        expanded_texts: List[str] = []
        expanded_ids: List[str] = []
        expanded_pref_labels: List[str] = []

        for _, r in df.iterrows():
            alts = [a.strip() for a in str(r["altLabels"]).split("\n") if a and a.strip()]
            if not alts:
                continue
            expanded_texts.extend(alts)
            expanded_ids.extend([r["conceptUri"]] * len(alts))
            expanded_pref_labels.extend([r["preferredLabel"]] * len(alts))

        if not expanded_texts:
            # No alt labels available; create empty cache
            self.alt_vecs[lang] = np.zeros((0, 768), dtype=np.float32)  # dimensionality doesn't matter here
            self.alt_ids[lang] = []
            self.alt_labels[lang] = []
            self.alt_texts[lang] = []
            return self.alt_vecs[lang], self.alt_ids[lang], self.alt_labels[lang]

        emb = self._embedder(lang)
        print(f"   ↳ encoding {len(expanded_texts):,} altLabels for {lang}")
        V_alt = emb.encode(expanded_texts, batch_size=128,
                           normalize_embeddings=True, convert_to_numpy=True,
                           show_progress_bar=True)
        self.alt_vecs[lang]   = V_alt
        self.alt_ids[lang]    = expanded_ids
        self.alt_labels[lang] = expanded_pref_labels  # keep preferredLabel for consistency
        self.alt_texts[lang]  = expanded_texts
        return V_alt, self.alt_ids[lang], self.alt_labels[lang]

    def retrieve_by_altlabels(self, lang: str, text: str, top_k: int | None = None, threshold: float = 0.50) -> pd.DataFrame:
        """
        Retrieve skills using altLabels as the candidate index.
        Returns a DataFrame with columns: skill_id, skill_label (preferredLabel).
        """
        V_alt, sids_alt, slabels_pref = self._skill_index_alt(lang)
        if V_alt.shape[0] == 0:
            return pd.DataFrame(columns=["skill_id", "skill_label"])

        emb = self._embedder(lang)
        sent_vec = emb.encode(
            _SENT_SPLIT.split(text.strip()),
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        sims = np.dot(sent_vec, V_alt.T)

        k = top_k if top_k is not None else self.k
        keep = min(k, V_alt.shape[0])
        best_idx = np.argpartition(-sims, keep-1, axis=1)[:, : keep]

        # Collect globally best unique concept IDs
        idx_score_pairs = []
        for row, idxs in zip(sims, best_idx):
            for j in idxs:
                idx_score_pairs.append((j, row[j]))

        idx_score_pairs.sort(key=lambda x: -x[1])
        seen, rows = set(), []
        for j, score in idx_score_pairs:
            if score < threshold:
                continue
            sid = sids_alt[j]
            if sid in seen:
                continue
            rows.append({"skill_id": sid, "skill_label": slabels_pref[j]})
            seen.add(sid)
            if len(rows) == k:
                break

        return pd.DataFrame(rows)


Define `extract_skills` — processes one Stage 1 output file using the preferredLabel index.

For each job posting:
1. Calls `retriever.retrieve()` to get top-K matching ESCO skills
2. Stores skill URIs as a comma-separated string in `skill_ids`
3. Stores skill labels as a comma-separated string in `skill_labels`
4. Sets empty strings if no skills exceed the similarity threshold

Saves the enriched DataFrame and updates the Stage 2 process tracker.

In [ ]:
def extract_skills(stage_process_file, row_index, retriever, output_path, stage_path):

    process_df = pd.read_pickle(stage_process_file.loc[row_index, "clean_path"])
    print(f'🔸 loaded {len(process_df):,} postings from {stage_process_file.loc[row_index,"clean_path"]}')

    for _, row in tqdm(process_df.iterrows(), total=len(process_df), desc="retrieving"):
        extr_skills_df = retriever.retrieve(row["desc_lang"], row["clean_desc"]).copy()

       # Check if required columns exist
        if 'skill_id' not in extr_skills_df.columns or 'skill_label' not in extr_skills_df.columns:
            # Set empty values if columns don't exist
            process_df.loc[_, "skill_ids"] = ''
            process_df.loc[_, "skill_labels"] = ''
            continue

        # Only proceed if we have valid data
        if not extr_skills_df.empty:
            process_df.loc[_, "skill_ids"] = ','.join(extr_skills_df["skill_id"].astype(str))
            process_df.loc[_, "skill_labels"] = ','.join(extr_skills_df["skill_label"].astype(str))
        else:
            process_df.loc[_, "skill_ids"] = ''
            process_df.loc[_, "skill_labels"] = ''

    extract_skills_path = os.path.join(output_path, f"{stage_process_file.loc[row_index,'input_file']}.pkl")

    process_df.to_pickle(extract_skills_path)

    stage_process_file.loc[row_index,"extract_path"] = extract_skills_path
    stage_process_file.loc[row_index,"extract_status"] = "complete"
    stage_process_file.to_pickle(stage_path)
    return stage_process_file

Define `extract_alt_skills` — same as `extract_skills` but uses the altLabels index. Matching against alternative labels broadens recall for skills that are rarely mentioned by their primary ESCO name. Results are stored in `skill_alt_ids` and `skill_alt_labels` columns.

In [ ]:
def extract_alt_skills(stage_process_file, row_index, retriever, output_path, stage_path):

    process_df = pd.read_pickle(stage_process_file.loc[row_index, "clean_path"])
    print(f'🔸 loaded {len(process_df):,} postings from {stage_process_file.loc[row_index,"clean_path"]}')

    for _, row in tqdm(process_df.iterrows(), total=len(process_df), desc="retrieving"):
        extr_skills_df = retriever.retrieve_by_altlabels(row["desc_lang"], row["clean_desc"]).copy()

       # Check if required columns exist
        if 'skill_alt_id' not in extr_skills_df.columns or 'skill_alt_label' not in extr_skills_df.columns:
            # Set empty values if columns don't exist
            process_df.loc[_, "skill_alt_ids"] = ''
            process_df.loc[_, "skill_alt_labels"] = ''
            continue

        # Only proceed if we have valid data
        if not extr_skills_df.empty:
            process_df.loc[_, "skill_alt_ids"] = ','.join(extr_skills_df["skill_alt_id"].astype(str))
            process_df.loc[_, "skill_alt_labels"] = ','.join(extr_skills_df["skill_alt_label"].astype(str))
        else:
            process_df.loc[_, "skill_alt_ids"] = ''
            process_df.loc[_, "skill_alt_labels"] = ''

    extract_skills_path = os.path.join(output_path, f"{stage_process_file.loc[row_index,'input_file']}.pkl")

    process_df.to_pickle(extract_skills_path)

    stage_process_file.loc[row_index,"extract_alt_path"] = extract_skills_path
    stage_process_file.loc[row_index,"extract_alt_status"] = "complete"
    stage_process_file.to_pickle(stage_path)
    return stage_process_file

## 2.2. Loading model for skills extraction

The multilingual sentence-transformer model is loaded automatically by `SkillRetriever` and cached in the standard Hugging Face cache. No model files are written inside the replication package.

> **Note:** First run requires an internet connection and downloads ~900 MB.

In [ ]:
# SkillRetriever loads and caches the configured SentenceTransformer model on first use.

Initialise the `SkillRetriever` with the skills folder path from `.env` and `top_k=20` (return at most 20 skills per posting). The skill index is built lazily on first use in the extraction loop below.

In [ ]:
skills_retriever = SkillRetriever(g.Config.SKILLS_PATH, 20)

## 2.3. Skills extraction and log saving

Build or update the Stage 2 process tracker by syncing with the Stage 1 tracker. Displays the tracker to confirm which files are registered and their processing status.

In [ ]:
stage_process_df = get_stage_process_df(g.Config.STAGE1_PROCESS_PATH, g.Config.STAGE2_PROCESS_PATH)
stage_process_df

Ensure the Stage 2 output folder exists before writing any files.

In [ ]:
g.check_folder_exists(g.Config.STAGE2_OUTPUT_PATH)

Main extraction loop. Iterates over all registered files and runs `extract_skills` on any that have not yet been processed (`extract_status != complete`). Each completed file is immediately marked in the tracker so the loop can be safely interrupted and resumed.

In [ ]:
for _, row in stage_process_df.iterrows():
    if pd.isna(row["extract_status"]) or row["extract_status"] != "complete":
        stage_process_df = extract_skills(stage_process_df, _, skills_retriever, g.Config.STAGE2_OUTPUT_PATH, g.Config.STAGE2_PROCESS_PATH)

Verification cell. Load the output pickle for the demo file and inspect the first rows to confirm `skill_ids` and `skill_labels` columns are populated.

In [ ]:
pd.read_pickle(os.path.join(g.Config.STAGE2_OUTPUT_PATH, "ua-2024-01-01.pkl"))

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.